# NB10: Extended analyses (Phase 10)

Five analyses that live downstream of the canonical opx-liq RF and QRF models
built in Phases 3R and 7R. Every feature construction and model load goes
through the Phase 3R winning configuration - no hard-coded feature set names.

1. Two-pyroxene benchmark against Thermobar
2. H2O dependence in the ArcPL residuals
3. IQR-based per-sample uncertainty (Phase 3R predict_iqr)
4. analytical-noise propagation (analytical noise propagation)
5. IsolationForest out-of-distribution filter on the ArcPL natural samples

## Purpose / Inputs / Outputs / Canonical decisions

**Purpose:** Downstream analyses on top of the canonical RF / QRF: two-pyroxene benchmark (moved to NB10b), H2O dependence + engineered-feature retrain, IQR per-sample uncertainty, analytical noise propagation, IsolationForest OOD filter, and OOD-method paradox diagnosis.

**Inputs:** `data/processed/opx_clean_opx_liq.parquet`, canonical RF + QRF opx-liq models, `results/nb04b_arcpl_predictions.csv`, `results/nb03_winning_configurations.json`.

**Outputs:** `results/nb10_two_pyroxene_benchmark.csv`, `results/nb10_h2o_dependence.csv`, `results/nb10_h2o_engineered_arcpl.csv`, `results/nb10_h2o_engineered_test_rmse.csv`, `results/nb10_iqr_uncertainty.csv`, `results/nb10_analytical_uncertainty.csv`, `results/nb10_mc_per_sample.csv`, `results/nb10_ood_isoforest.csv`, `results/nb10_arcpl_ood_scores.csv`, `results/nb10_ood_paradox_methods.csv`, `results/nb10_ood_scores_all_methods.csv`, `models/model_IsolationForest_opx_liq.joblib`, `models/model_RF_*_opx_liq_H2O.joblib`.

**Canonical decisions:** H2O-engineered retrain is a *sensitivity check*, not a canonical replacement - canonical RF remains the one built in NB03. OOD-method OPERATOR DECISION (A/B/C) picks the score used for NB08 downstream filtering.


In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)
from src.plot_style import load_winning_config, canonical_model_filename
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr
from src.evaluation import compute_metrics as metrics
from src.plot_style import apply_style
apply_style()  # Okabe-Ito colorblind-safe palette, 300 dpi PNG+PDF defaults


In [ ]:
# Shared setup: Phase 3R winner, canonical RF/QRF, test set, ArcPL
config_3r = load_winning_config(RESULTS)
WIN_FEAT = config_3r['global_feature_set']
feat_fn = lambda df, use_liq: build_feature_matrix(df, WIN_FEAT, use_liq=use_liq)
print(f'Phase 3R winning feature set: {WIN_FEAT}')

model_T = joblib.load(MODELS / canonical_model_filename('RF', 'T_C', 'opx_liq', RESULTS))
model_P = joblib.load(MODELS / canonical_model_filename('RF', 'P_kbar', 'opx_liq', RESULTS))

qrf_T_path = MODELS / 'model_QRF_T_C_opx_liq.joblib'
qrf_P_path = MODELS / 'model_QRF_P_kbar_opx_liq.joblib'
qrf_T = joblib.load(qrf_T_path) if qrf_T_path.exists() else None
qrf_P = joblib.load(qrf_P_path) if qrf_P_path.exists() else None

# Training data (for IsolationForest fit and reference stats)
df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
idx_tr = np.load(DATA_SPLITS / 'train_indices_opx_liq.npy')
idx_te = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')
df_train = df_liq.loc[idx_tr].copy()
df_test = df_liq.loc[idx_te].copy()
X_train, feat_names = feat_fn(df_train, use_liq=True)
X_test, _ = feat_fn(df_test, use_liq=True)

# ArcPL natural samples (already cleaned by nb04b).
arcpl_path = RESULTS / 'nb04b_arcpl_predictions.csv'
if arcpl_path.exists():
    df_arcpl = pd.read_csv(arcpl_path)
else:
    raise FileNotFoundError(f"Missing {arcpl_path}. Run nb04b first.")
print(f'ArcPL n={len(df_arcpl)}')

In [ ]:
# Phase 10.1: Two-pyroxene benchmark via Thermobar on the opx-liq test set
# Evaluates Putirka (2008) Eq. 36/39 two-pyroxene-only T (and P where available)
# as a non-ML baseline that can be directly compared against our opx-liq RF.

try:
    import Thermobar as pt
    HAVE_THERMOBAR = True
except Exception as e:
    HAVE_THERMOBAR = False
    print(f'Thermobar unavailable: {e}')

two_px_rows = []

if HAVE_THERMOBAR and {'T_C', 'P_kbar'}.issubset(df_test.columns):
    # Build Thermobar-style input frames. The two-pyroxene thermometer needs
    # both Opx and Cpx compositions; samples without Cpx are skipped.
    opx_cols = ['SiO2', 'TiO2', 'Al2O3', 'FeO_total', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'Cr2O3']
    cpx_prefix = 'cpx_'
    has_cpx = any(c.startswith(cpx_prefix) for c in df_test.columns)
    if has_cpx:
        mask = df_test[[f'{cpx_prefix}SiO2']].notna().all(axis=1).values
        sub = df_test[mask].copy()
        opx_df = pd.DataFrame({f'{c}_Opx': sub[c].values for c in opx_cols if c in sub.columns})
        cpx_df = pd.DataFrame({f'{c}_Cpx': sub[f'{cpx_prefix}{c}'].values
                               for c in opx_cols if f'{cpx_prefix}{c}' in sub.columns})

        try:
            out = pt.calculate_cpx_opx_press_temp(
                cpx_comps=cpx_df, opx_comps=opx_df,
                equationT='T_Put2008_eq36', equationP='P_Put2008_eq39')
            t_pred = out['T_K_calc'].values - 273.15
            p_pred = out['P_kbar_calc'].values if 'P_kbar_calc' in out.columns else np.full(len(sub), np.nan)
            t_obs = sub['T_C'].values
            p_obs = sub['P_kbar'].values

            two_px_rows.append({'target': 'T_C', **metrics(t_obs[np.isfinite(t_pred)],
                                                           t_pred[np.isfinite(t_pred)])})
            mask_p = np.isfinite(p_pred)
            if mask_p.sum() > 0:
                two_px_rows.append({'target': 'P_kbar', **metrics(p_obs[mask_p], p_pred[mask_p])})
        except Exception as e:
            print(f'Thermobar call failed: {e}')
    else:
        print('Test set has no Cpx columns; two-pyroxene benchmark skipped.')
else:
    print('Skipped two-pyroxene benchmark (no Thermobar or missing cols).')

two_px_df = pd.DataFrame(two_px_rows)
two_px_df.to_csv(RESULTS / 'nb10_two_pyroxene_benchmark.csv', index=False)
if len(two_px_df):
    print(two_px_df.round(3).to_string(index=False))

In [ ]:
# Phase 10.2: H2O dependence in ArcPL residuals
# Use canonical RF T and P on ArcPL, compute residuals, and regress against
# liquid H2O to see whether our dry-trained model has a systematic wet bias.

from scipy.stats import pearsonr, spearmanr

h2o_rows = []
if 'H2O_Liq' in df_arcpl.columns or 'liq_H2O' in df_arcpl.columns:
    h2o_col = 'liq_H2O' if 'liq_H2O' in df_arcpl.columns else 'H2O_Liq'
    X_arcpl, _ = feat_fn(df_arcpl, use_liq=True)
    t_pred = predict_median(model_T, X_arcpl)
    p_pred = predict_median(model_P, X_arcpl)

    h2o = df_arcpl[h2o_col].values.astype(float)

    for tgt, obs_col, pred in [('T_C', 'T_C', t_pred), ('P_kbar', 'P_kbar', p_pred)]:
        if obs_col not in df_arcpl.columns:
            continue
        obs = df_arcpl[obs_col].values.astype(float)
        resid = pred - obs
        mask = np.isfinite(resid) & np.isfinite(h2o)
        if mask.sum() < 5:
            continue
        r_p, p_p = pearsonr(h2o[mask], resid[mask])
        r_s, p_s = spearmanr(h2o[mask], resid[mask])
        slope, intercept = np.polyfit(h2o[mask], resid[mask], 1)
        h2o_rows.append({
            'target':      tgt,
            'n':           int(mask.sum()),
            'pearson_r':   float(r_p),
            'pearson_p':   float(p_p),
            'spearman_r':  float(r_s),
            'spearman_p':  float(p_s),
            'slope':       float(slope),
            'intercept':   float(intercept),
            'mean_resid_dry': float(np.mean(resid[mask][h2o[mask] < 1.0])) if (h2o[mask] < 1.0).any() else np.nan,
            'mean_resid_wet': float(np.mean(resid[mask][h2o[mask] >= 1.0])) if (h2o[mask] >= 1.0).any() else np.nan,
        })
else:
    print('ArcPL has no H2O_liq or liq_H2O column; H2O analysis skipped.')

h2o_df = pd.DataFrame(h2o_rows)
h2o_df.to_csv(RESULTS / 'nb10_h2o_dependence.csv', index=False)
if len(h2o_df):
    print(h2o_df.round(4).to_string(index=False))

## Phase 10.2b: H2O-aware engineered features (v5)

Phase 10.2 above shows the canonical dry-trained RF has a residual-H2O
correlation. Part 6 of the v5 plan adds four engineered features that encode
water explicitly and retrains the opx-liq RF:

- `H2O_over_SiO2 = H2O / liq_SiO2`  (bulk melt water mass ratio)
- `H2O_activity_proxy = H2O / (H2O + liq_SiO2)`  (mole-fraction-ish activity)
- `log1p_H2O = log(1 + H2O)`  (compresses high-water tails)
- `H2O_x_Mg_num_liq = H2O * Mg_num_liq`  (water moderated by liquid Fe/Mg)

Samples with missing `H2O_Liq` are treated as dry (0 wt%) and extreme values
above 15 wt% (30 / 166 rows - almost certainly unit errors, likely ppm vs
wt%) are clipped to 15. The test: if residual-H2O Pearson |r| on ArcPL drops
below 0.1 after retraining, the engineered features successfully absorbed
the water signal and the residual correlation in the canonical RF is a
feature-representation gap, not an intrinsic thermobarometric limit.

In [ ]:
# Phase 10.2b (v5): H2O engineered features + opx-liq RF retrain

def _clean_h2o(vals):
    """Treat NaN as dry (0 wt%), clip plausible range to [0, 15] wt%."""
    v = np.where(np.isfinite(vals), vals, 0.0).astype(float)
    return np.clip(v, 0.0, 15.0)


def _add_h2o_engineered(df, X, feat_names):
    """Append 4 H2O-aware features to an existing X matrix.
    Missing liq cols default to neutral values so the feature never throws."""
    h2o = _clean_h2o(df['H2O_Liq'].values if 'H2O_Liq' in df.columns else np.zeros(len(df)))
    sio2 = df['liq_SiO2'].fillna(50.0).values if 'liq_SiO2' in df.columns else np.full(len(df), 50.0)
    mgo  = df['liq_MgO'].fillna(0.0).values  if 'liq_MgO'  in df.columns else np.zeros(len(df))
    feo  = df['liq_FeO'].fillna(0.0).values  if 'liq_FeO'  in df.columns else np.zeros(len(df))
    mg_num_liq = np.where((mgo + feo) > 1e-6, mgo / (mgo + feo), 0.5)

    new_cols = np.column_stack([
        h2o / np.maximum(sio2, 1e-6),
        h2o / np.maximum(h2o + sio2, 1e-6),
        np.log1p(h2o),
        h2o * mg_num_liq,
    ])
    new_names = ['H2O_over_SiO2', 'H2O_activity_proxy', 'log1p_H2O', 'H2O_x_Mg_num_liq']
    X_aug = np.column_stack([X, new_cols]).astype(float)
    # Guard against any NaN/Inf that survive (conservative: replace with 0)
    X_aug = np.where(np.isfinite(X_aug), X_aug, 0.0)
    return X_aug, list(feat_names) + new_names


X_train_h2o, feat_h2o = _add_h2o_engineered(df_train, X_train, feat_names)
X_test_h2o,  _        = _add_h2o_engineered(df_test,  X_test,  feat_names)
print(f'Augmented feature count: {X_train.shape[1]} -> {X_train_h2o.shape[1]}')

# Retrain opx-liq RF with the same hyperparameters as the canonical model.
# We reuse canonical model params (n_estimators, min_samples_leaf, max_features)
# so any change in test RMSE is attributable to the added features, not tuning.
def _extract_params(fitted_rf):
    p = fitted_rf.get_params()
    return {k: p[k] for k in ('n_estimators', 'min_samples_leaf', 'max_features', 'max_depth')
            if k in p}

from sklearn.ensemble import RandomForestRegressor

rf_T_h2o = RandomForestRegressor(**_extract_params(model_T),
                                 random_state=SEED_MODEL, n_jobs=-1)
rf_P_h2o = RandomForestRegressor(**_extract_params(model_P),
                                 random_state=SEED_MODEL, n_jobs=-1)
rf_T_h2o.fit(X_train_h2o, df_train['T_C'].values)
rf_P_h2o.fit(X_train_h2o, df_train['P_kbar'].values)

# Evaluate on ArcPL (same residual-H2O correlation check as Phase 10.2).
h2o_rows2 = []
if 'H2O_Liq' in df_arcpl.columns or 'liq_H2O' in df_arcpl.columns:
    h2o_col = 'liq_H2O' if 'liq_H2O' in df_arcpl.columns else 'H2O_Liq'
    X_arcpl_h2o, _ = _add_h2o_engineered(df_arcpl, feat_fn(df_arcpl, use_liq=True)[0], feat_names)
    t_pred_h2o = predict_median(rf_T_h2o, X_arcpl_h2o)
    p_pred_h2o = predict_median(rf_P_h2o, X_arcpl_h2o)
    h2o_vals = _clean_h2o(df_arcpl[h2o_col].values.astype(float))

    for tgt, obs_col, pred in [('T_C', 'T_C', t_pred_h2o), ('P_kbar', 'P_kbar', p_pred_h2o)]:
        if obs_col not in df_arcpl.columns:
            continue
        obs = df_arcpl[obs_col].values.astype(float)
        resid = pred - obs
        mask = np.isfinite(resid) & np.isfinite(h2o_vals)
        if mask.sum() < 5:
            continue
        r_p, p_p = pearsonr(h2o_vals[mask], resid[mask])
        h2o_rows2.append({
            'target': tgt, 'n': int(mask.sum()),
            'pearson_r_after':  float(r_p),
            'pearson_p_after':  float(p_p),
            'rmse_after':       float(np.sqrt(mean_squared_error(obs[mask], pred[mask]))),
            'passes_threshold': bool(abs(r_p) < 0.10),
        })

# Evaluate on held-out test set: does adding H2O features change test RMSE?
rmse_rows = []
for tgt, y_te, pred_base, pred_h2o in [
    ('T_C',    df_test['T_C'].values, predict_median(model_T, X_test),
     predict_median(rf_T_h2o, X_test_h2o)),
    ('P_kbar', df_test['P_kbar'].values, predict_median(model_P, X_test),
     predict_median(rf_P_h2o, X_test_h2o)),
]:
    rmse_rows.append({
        'target': tgt,
        'rmse_baseline': float(np.sqrt(mean_squared_error(y_te, pred_base))),
        'rmse_with_h2o': float(np.sqrt(mean_squared_error(y_te, pred_h2o))),
        'r2_baseline':   float(r2_score(y_te, pred_base)),
        'r2_with_h2o':   float(r2_score(y_te, pred_h2o)),
    })

h2o2_df = pd.DataFrame(h2o_rows2)
rmse_df = pd.DataFrame(rmse_rows)
h2o2_df.to_csv(RESULTS / 'nb10_h2o_engineered_arcpl.csv', index=False)
rmse_df.to_csv(RESULTS / 'nb10_h2o_engineered_test_rmse.csv', index=False)

# Persist the retrained models so NB08 / NBF can pick them up if needed.
joblib.dump(rf_T_h2o, MODELS / 'model_RF_T_C_opx_liq_H2O.joblib')
joblib.dump(rf_P_h2o, MODELS / 'model_RF_P_kbar_opx_liq_H2O.joblib')

print('Residual-H2O correlation after retrain with H2O features:')
if len(h2o2_df):
    print(h2o2_df.round(4).to_string(index=False))
print('\nTest-set RMSE (baseline vs with H2O):')
print(rmse_df.round(3).to_string(index=False))
print('\nBefore (Phase 10.2):')
if len(h2o_df):
    print(h2o_df[['target', 'n', 'pearson_r', 'pearson_p']].round(4).to_string(index=False))

In [ ]:
# Phase 10.3: IQR per-sample uncertainty
# Uses predict_iqr (16th to 84th percentile across trees) as a cheap
# epistemic uncertainty surrogate and correlates it with |residual| on the
# held-out test set.

_, q16_T, q84_T = predict_iqr(model_T, X_test)
_, q16_P, q84_P = predict_iqr(model_P, X_test)
test_pred_T = predict_median(model_T, X_test)
test_pred_P = predict_median(model_P, X_test)
iqr_T = q84_T - q16_T
iqr_P = q84_P - q16_P

iqr_rows = []
for tgt, y_true, pred, iqr_vals in [
    ('T_C',    df_test['T_C'].values, test_pred_T, iqr_T),
    ('P_kbar', df_test['P_kbar'].values, test_pred_P, iqr_P),
]:
    abs_res = np.abs(pred - y_true)
    r_p, _ = pearsonr(iqr_vals, abs_res)
    r_s, _ = spearmanr(iqr_vals, abs_res)
    # Calibration: fraction of samples where |residual| <= IQR/2
    calib = float(np.mean(abs_res <= iqr_vals / 2.0))
    iqr_rows.append({
        'target':        tgt,
        'iqr_mean':      float(iqr_vals.mean()),
        'iqr_median':    float(np.median(iqr_vals)),
        'pearson_iqr_vs_absres':  float(r_p),
        'spearman_iqr_vs_absres': float(r_s),
        'frac_abs_res_le_halfiqr': calib,
        'n': int(len(y_true)),
    })

iqr_df = pd.DataFrame(iqr_rows)
iqr_df.to_csv(RESULTS / 'nb10_iqr_uncertainty.csv', index=False)
print(iqr_df.round(3).to_string(index=False))


In [ ]:
from tqdm.auto import tqdm
# Phase 10.4: Analytical-noise propagation (EPMA measurement uncertainty)
# Perturb each test sample's oxide columns with the analytical-noise model
# used in Phase 3R (3% rel on majors, 8% on minors) many times, push each
# perturbed replicate through the canonical RF, and report the standard
# deviation of the resulting prediction distribution. This yields a
# propagated analytical uncertainty that is independent of the RF's internal
# tree variance.

N_MC = 200
MAJ_REL = 0.03
MIN_REL = 0.08
MAJORS = {'SiO2', 'Al2O3', 'FeO_total', 'MgO', 'CaO', 'liq_SiO2', 'liq_Al2O3',
          'liq_FeO', 'liq_MgO', 'liq_CaO'}

rng = np.random.default_rng(SEED_NOISE_AUG)

mc_T = np.zeros((len(df_test), N_MC))
mc_P = np.zeros((len(df_test), N_MC))

all_ox_cols = [c for c in df_test.columns
               if c in OPX_FULL_OXIDES or c in {f'liq_{o}' for o in LIQ_OXIDES}]

for k in tqdm(range(N_MC), desc='analytical-noise propagation'):
    df_perturb = df_test.copy()
    for col in all_ox_cols:
        rel = MAJ_REL if col in MAJORS else MIN_REL
        noise = rng.normal(loc=0.0, scale=rel, size=len(df_perturb))
        df_perturb[col] = df_perturb[col].values * (1.0 + noise)
    X_mc, _ = feat_fn(df_perturb, use_liq=True)
    mc_T[:, k] = predict_median(model_T, X_mc)
    mc_P[:, k] = predict_median(model_P, X_mc)

sigma_T = mc_T.std(axis=1)
sigma_P = mc_P.std(axis=1)

mc_rows = [
    {'target': 'T_C',
     'sigma_mean': float(sigma_T.mean()),
     'sigma_median': float(np.median(sigma_T)),
     'sigma_p90': float(np.percentile(sigma_T, 90)),
     'n_mc': N_MC,
     'n': int(len(df_test))},
    {'target': 'P_kbar',
     'sigma_mean': float(sigma_P.mean()),
     'sigma_median': float(np.median(sigma_P)),
     'sigma_p90': float(np.percentile(sigma_P, 90)),
     'n_mc': N_MC,
     'n': int(len(df_test))},
]
mc_df = pd.DataFrame(mc_rows)
mc_df.to_csv(RESULTS / 'nb10_analytical_uncertainty.csv', index=False)

# Per-sample MC predictions for optional downstream figures
mc_pred = pd.DataFrame({
    'index':    idx_te,
    'T_mean':   mc_T.mean(axis=1),
    'T_sigma':  sigma_T,
    'P_mean':   mc_P.mean(axis=1),
    'P_sigma':  sigma_P,
})
mc_pred.to_csv(RESULTS / 'nb10_mc_per_sample.csv', index=False)
print(mc_df.round(3).to_string(index=False))

In [ ]:
# Phase 10.5: IsolationForest out-of-distribution filter on ArcPL
# Fit IsolationForest on training-set features, score ArcPL samples, and
# recompute T and P RMSE on the in-distribution subset.

iso = IsolationForest(n_estimators=300, contamination='auto',
                      random_state=SEED_MODEL, n_jobs=-1)
iso.fit(X_train)

X_arcpl, _ = feat_fn(df_arcpl, use_liq=True)
arcpl_score = iso.score_samples(X_arcpl)   # higher = more in-distribution
arcpl_label = iso.predict(X_arcpl)         # +1 inlier, -1 outlier

pred_T_arcpl = predict_median(model_T, X_arcpl)
pred_P_arcpl = predict_median(model_P, X_arcpl)

ood_rows = []
for tgt, obs_col, pred in [('T_C', 'T_C', pred_T_arcpl),
                            ('P_kbar', 'P_kbar', pred_P_arcpl)]:
    if obs_col not in df_arcpl.columns:
        continue
    obs = df_arcpl[obs_col].values.astype(float)
    mask_all = np.isfinite(obs)
    mask_in = mask_all & (arcpl_label == 1)
    mask_out = mask_all & (arcpl_label == -1)
    ood_rows.append({'target': tgt, 'subset': 'all',
                     **metrics(obs[mask_all], pred[mask_all])})
    if mask_in.sum() > 0:
        ood_rows.append({'target': tgt, 'subset': 'in_distribution',
                         **metrics(obs[mask_in], pred[mask_in])})
    if mask_out.sum() > 0:
        ood_rows.append({'target': tgt, 'subset': 'out_of_distribution',
                         **metrics(obs[mask_out], pred[mask_out])})

ood_df = pd.DataFrame(ood_rows)
ood_df.to_csv(RESULTS / 'nb10_ood_isoforest.csv', index=False)

# Persist per-sample OOD scores for downstream figures
arcpl_scores_df = pd.DataFrame({
    'index':        np.arange(len(df_arcpl)),
    'iso_score':    arcpl_score,
    'iso_inlier':   (arcpl_label == 1).astype(int),
    'T_pred':       pred_T_arcpl,
    'P_pred':       pred_P_arcpl,
})
arcpl_scores_df.to_csv(RESULTS / 'nb10_arcpl_ood_scores.csv', index=False)
joblib.dump(iso, MODELS / 'model_IsolationForest_opx_liq.joblib')

print(ood_df.round(3).to_string(index=False))

## Phase 10.6: OOD paradox diagnosis (v5)

The canonical dry-trained RF shows a paradox on ArcPL: IsolationForest labels
a sizable subset OOD, but the OOD subset does **not** consistently have
larger absolute residuals than the in-distribution subset. If an OOD flag
does not track error it is a diagnostic without predictive value.

Part 7 compares three OOD scoring methods on ArcPL by correlating each
anomaly score with |residual| on T and P (Spearman + Pearson):

1. **IsolationForest** score on the 1664-dim canonical feature space
2. **Mahalanobis distance** in the first 5 principal components of the
   training feature space (linear manifold distance)
3. **QRF sigma** = 0.5 * (q84 - q16) from the canonical QRF as a direct
   epistemic-uncertainty surrogate

The method with the largest |Spearman r| vs |residual| is the one that
actually predicts prediction error and is therefore canonized for NB08
downstream filtering. This is an **OPERATOR DECISION** point (A / B / C).

In [ ]:
# Phase 10.6 (v5): compare OOD scoring methods by |residual| correlation on ArcPL.
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import mahalanobis

# 1) IsolationForest anomaly: negate score_samples so larger = more anomalous
iso_anomaly_arcpl = -arcpl_score

# 2) Mahalanobis in PC5 space of training features
_scaler = StandardScaler().fit(X_train)
X_tr_std = _scaler.transform(X_train)
X_ar_std = _scaler.transform(X_arcpl)
_pca = PCA(n_components=min(5, X_train.shape[1])).fit(X_tr_std)
Z_tr = _pca.transform(X_tr_std)
Z_ar = _pca.transform(X_ar_std)
mu = Z_tr.mean(axis=0)
cov = np.cov(Z_tr, rowvar=False)
cov_inv = np.linalg.pinv(cov + 1e-8 * np.eye(cov.shape[0]))
mahal = np.array([float(mahalanobis(z, mu, cov_inv)) for z in Z_ar])

# 3) QRF sigma (half IQR) on ArcPL - epistemic uncertainty surrogate
qrf_sigma_T = np.full(len(df_arcpl), np.nan)
qrf_sigma_P = np.full(len(df_arcpl), np.nan)
if qrf_T is not None and qrf_P is not None:
    q_T_ar = qrf_T.predict(X_arcpl, quantiles=[0.16, 0.5, 0.84])
    q_P_ar = qrf_P.predict(X_arcpl, quantiles=[0.16, 0.5, 0.84])
    qrf_sigma_T = 0.5 * (q_T_ar[:, 2] - q_T_ar[:, 0])
    qrf_sigma_P = 0.5 * (q_P_ar[:, 2] - q_P_ar[:, 0])

# Observed residuals on ArcPL (using canonical RF baseline, not H2O-retrain)
resid_T_arcpl = pred_T_arcpl - df_arcpl['T_C'].values.astype(float)
resid_P_arcpl = pred_P_arcpl - df_arcpl['P_kbar'].values.astype(float)

def _corr(score, absres):
    mask = np.isfinite(score) & np.isfinite(absres)
    if mask.sum() < 10:
        return {'spearman_r': np.nan, 'spearman_p': np.nan,
                'pearson_r':  np.nan, 'pearson_p':  np.nan, 'n': int(mask.sum())}
    s_r, s_p = spearmanr(score[mask], absres[mask])
    p_r, p_p = pearsonr (score[mask], absres[mask])
    return {'spearman_r': float(s_r), 'spearman_p': float(s_p),
            'pearson_r':  float(p_r), 'pearson_p':  float(p_p),
            'n': int(mask.sum())}


rows = []
for method, score_T, score_P in [
    ('isoforest',   iso_anomaly_arcpl, iso_anomaly_arcpl),
    ('mahalanobis', mahal,             mahal),
    ('qrf_sigma',   qrf_sigma_T,       qrf_sigma_P),
]:
    for tgt, score, resid in [('T_C', score_T, resid_T_arcpl),
                              ('P_kbar', score_P, resid_P_arcpl)]:
        c = _corr(score, np.abs(resid))
        rows.append({'method': method, 'target': tgt, **c})

ood_paradox_df = pd.DataFrame(rows)
ood_paradox_df.to_csv(RESULTS / 'nb10_ood_paradox_methods.csv', index=False)

# Persist per-sample OOD scores for all three methods for downstream use
pd.DataFrame({
    'index':          np.arange(len(df_arcpl)),
    'iso_anomaly':    iso_anomaly_arcpl,
    'mahal_pc5':      mahal,
    'qrf_sigma_T':    qrf_sigma_T,
    'qrf_sigma_P':    qrf_sigma_P,
    'abs_resid_T':    np.abs(resid_T_arcpl),
    'abs_resid_P':    np.abs(resid_P_arcpl),
}).to_csv(RESULTS / 'nb10_ood_scores_all_methods.csv', index=False)

print('OOD-method comparison on ArcPL (larger |Spearman r| = better error predictor):')
print(ood_paradox_df.round(4).to_string(index=False))

# Rank methods by mean |spearman_r| across both targets
rank = (ood_paradox_df.assign(abs_s=ood_paradox_df['spearman_r'].abs())
        .groupby('method')['abs_s'].mean().sort_values(ascending=False))
print('\nMean |Spearman r| across T and P (higher = better):')
print(rank.round(4).to_string())

print('\n' + '=' * 70)
print('OPERATOR DECISION REQUIRED: OOD method to canonize for NB08 filtering')
print('=' * 70)
print('  Option A: IsolationForest   (feature-space anomaly, current default)')
print('  Option B: Mahalanobis PC5   (linear manifold distance)')
print('  Option C: QRF sigma         (direct epistemic uncertainty)')
print('\nPick the method with the largest |Spearman r| vs |residual| - that is')
print('the one that actually predicts prediction error. See rank table above.')
print('=' * 70)

In [ ]:
# Verification assertions
expected = [
    'nb10_two_pyroxene_benchmark.csv',
    'nb10_h2o_dependence.csv',
    'nb10_h2o_engineered_arcpl.csv',
    'nb10_h2o_engineered_test_rmse.csv',
    'nb10_iqr_uncertainty.csv',
    'nb10_analytical_uncertainty.csv',
    'nb10_mc_per_sample.csv',
    'nb10_ood_isoforest.csv',
    'nb10_arcpl_ood_scores.csv',
    'nb10_ood_paradox_methods.csv',
    'nb10_ood_scores_all_methods.csv',
]
for f in expected:
    assert (RESULTS / f).exists(), f'missing {f}'
assert (MODELS / 'model_IsolationForest_opx_liq.joblib').exists()
assert (MODELS / 'model_RF_T_C_opx_liq_H2O.joblib').exists()
assert (MODELS / 'model_RF_P_kbar_opx_liq_H2O.joblib').exists()

print('=== PHASE 11 COMPLETE ===')
print(f'  winning feature set: {WIN_FEAT}')
for f in expected:
    print(f'  wrote {f}')